# Neural Machine Translation Demo with Marian
## Transformer Architecture

**By Ye Kyaw Thu, Lab Leader, Language Understanding Lab., Myanmar**  
**Date:** 26 May 2026  
*For AI (Fundamental) Class students*  

အရင်ဆုံး GPU အားမအား စစ်ပါ။  
(ဆရာ သုံးခဲ့တဲ့ GPU information ကို သိစေချင်လို့ run ပြတာလည်း ပါပါတယ်)

In [1]:
%pwd

'/home/phantom/Documents/assignment_7_testing/kaung-htet-htun_assignment-7/notebooks'

In [2]:
!nvidia-smi

Fri Jul  3 15:35:17 2026       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.309.01             Driver Version: 535.309.01   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA GeForce GTX 1050 ...    Off | 00000000:03:00.0 Off |                  N/A |
| N/A   46C    P8              N/A / ERR! |      4MiB /  4096MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

## Shell Script Preparation for Tranformer Architecture

Vocab ဖိုင်က ရှိပြီးသား ဖြစ်ရပါမယ်။  

In [4]:
from IPython.display import Markdown

# Read the file content
with open('../transformer.phmy.sh', 'r') as f:
    content = f.read()

# Display it as a highlighted Bash block
display(Markdown(f"```bash\n{content}\n```"))

```bash
#!/bin/bash

## Written by Ye Kyaw Thu, Affiliated Professor, CADT, Cambodia
## for NMT Experiments between Burmese and Ethnic Languages
## used Marian NMT Framework for training
## Last updated: 23 May 2022


model_folder="model.transformer.phmy";
mkdir ${model_folder};
data_path="/home/phantom/Documents/assignment_7_testing/kaung-htet-htun_assignment-7/g2p-par";
src="ph"; tgt="my";

~/marian/build/marian \
    --model ${model_folder}/model.npz --type transformer \
    --train-sets ${data_path}/train.${src} ${data_path}/train.${tgt} \
    --max-length 200 \
    --vocabs ${data_path}/vocab/vocab.${src}.yml ${data_path}/vocab/vocab.${tgt}.yml \
    --mini-batch-fit -w 1000 --maxi-batch 100 \
    --early-stopping 10 \
    --valid-freq 5000 --save-freq 5000 --disp-freq 500 \
    --valid-metrics cross-entropy perplexity bleu \
    --valid-sets ${data_path}/dev.${src} ${data_path}/dev.${tgt} \
    --valid-translation-output ${model_folder}/valid.${src}-${tgt}.output --quiet-translation \
    --valid-mini-batch 64 \
    --beam-size 6 --normalize 0.6 \
    --log ${model_folder}/train.log --valid-log ${model_folder}/valid.log \
    --enc-depth 2 --dec-depth 2 \
    --transformer-heads 8 \
    --transformer-postprocess-emb d \
    --transformer-postprocess dan \
    --transformer-dropout 0.3 --label-smoothing 0.1 \
    --learn-rate 0.0003 --lr-warmup 0 --lr-decay-inv-sqrt 16000 --lr-report \
    --clip-norm 5 \
    --tied-embeddings \
    --devices 0 --sync-sgd --seed 1111 \
    --exponential-smoothing \
    --dump-config > ${model_folder}/${src}-${tgt}.config.yml
    
time ~/marian/build/marian -c ${model_folder}/${src}-${tgt}.config.yml  2>&1 | tee ${model_folder}/transformer-${src}-${tgt}.log
```

ဒီတခါတော့ architecture ကို transformer အဖြစ် ပြောင်းလိုက်ပါတယ်။  

```bash
--type transformer
```

## Training Transformer Model for Phoneme to Grapheme Translation

In [8]:
%pwd

'/home/phantom/Documents/assignment_7_testing/kaung-htet-htun_assignment-7/notebooks'

In [9]:
!../transformer.phmy.sh

mkdir: cannot create directory ‘model.transformer.phmy’: File exists
[2026-07-03 20:03:11] [marian] Marian v1.12.0 65bf82ff 2023-02-21 09:56:29 -0800
[2026-07-03 20:03:11] [marian] Running on PS42 as process 27690 with command line:
[2026-07-03 20:03:11] [marian] /home/phantom/marian/build/marian -c model.transformer.phmy/ph-my.config.yml
[2026-07-03 20:03:11] [config] after: 0e
[2026-07-03 20:03:11] [config] after-batches: 0
[2026-07-03 20:03:11] [config] after-epochs: 0
[2026-07-03 20:03:11] [config] all-caps-every: 0
[2026-07-03 20:03:11] [config] allow-unk: false
[2026-07-03 20:03:11] [config] authors: false
[2026-07-03 20:03:11] [config] beam-size: 4
[2026-07-03 20:03:11] [config] bert-class-symbol: "[CLS]"
[2026-07-03 20:03:11] [config] bert-mask-symbol: "[MASK]"
[2026-07-03 20:03:11] [config] bert-masking-fraction: 0.15
[2026-07-03 20:03:11] [config] bert-sep-symbol: "[SEP]"
[2026-07-03 20:03:11] [config] bert-train-type-embeddings: true
[2026-07-03 20:03:11] [config] bert-type-

Transformer မော်ဒယ်ကတော့ ၁၁မိနစ်ပဲ ကြာပါတယ်။  
Epoch 874 မှာ training ပြီးသွားပါတယ်။  
Validation ဒေတာနဲ့ အကောင်းဆုံး ရလဒ်ကတော့ 76.63 BLEU score ပါ။   

## Checking Models  

In [10]:
!ls ./model.transformer.phmy

model.iter10000.npz  model.iter50000.npz      model.npz.progress.yml
model.iter15000.npz  model.iter5000.npz       model.npz.yml
model.iter20000.npz  model.iter55000.npz      ph-my.config.yml
model.iter25000.npz  model.iter60000.npz      train.log
model.iter30000.npz  model.iter70000.npz      transformer-ph-my.log
model.iter35000.npz  model.npz		      valid.log
model.iter40000.npz  model.npz.decoder.yml    valid.ph-my.output
model.iter45000.npz  model.npz.optimizer.npz


In [11]:
!cat ./model.transformer.phmy/valid.log

[2026-07-03 16:08:52] [valid] Ep. 80 : Up. 5000 : cross-entropy : 1.83935 : new best
[2026-07-03 16:08:53] [valid] Ep. 80 : Up. 5000 : perplexity : 1.61375 : new best
[2026-07-03 16:08:53] [valid] First sentence's tokens as scored:
[2026-07-03 16:08:53] [valid] DefaultVocab keeps original segments for scoring
[2026-07-03 16:08:53] [valid]   Hyp: ဥတ် တ ရ ဖ လ ဂု နီ
[2026-07-03 16:08:53] [valid]   Ref: ဥတ် တ ရ ဖ လ ဂု နီ
[2026-07-03 16:08:55] [valid] Ep. 80 : Up. 5000 : bleu : 76.1047 : new best
[2026-07-03 16:28:31] [valid] Ep. 159 : Up. 10000 : cross-entropy : 2.11296 : stalled 1 times (last best: 1.83935)
[2026-07-03 16:28:31] [valid] Ep. 159 : Up. 10000 : perplexity : 1.73282 : stalled 1 times (last best: 1.61375)
[2026-07-03 16:28:34] [valid] Ep. 159 : Up. 10000 : bleu : 75.2163 : stalled 1 times (last best: 76.1047)
[2026-07-03 16:48:12] [valid] Ep. 239 : Up. 15000 : cross-entropy : 2.28532 : stalled 2 times (last best: 1.83935)
[2026-07-03 16:48:13] [valid] Ep. 239 : Up. 15000 : per

## Testing Transformer Model for Phoneme to Grapheme Translation

Marian ရဲ့ decoder command ကိုလည်း လေ့လာကြည့်ပါ။  

In [13]:
!~/marian/build/marian-decoder --help

Marian: Fast Neural Machine Translation in C++
Usage: /home/phantom/marian/build/marian-decoder [OPTIONS]

General options:
  -h,--help                             Print this help message and exit
  --version                             Print the version number and exit
  --authors                             Print list of authors and exit
  --cite                                Print citation and exit
  --build-info TEXT                     Print CMake build options and exit. Set to 'all' to print advanced options
  -c,--config VECTOR ...                Configuration file(s). If multiple, later overrides earlier
  -w,--workspace INT=512                Preallocate arg MB of work space. Negative `--workspace -N` value allocates workspace as total available GPU memory minus N megabytes.
  --log TEXT                            Log training process information to file given by arg
  --log-level TEXT=info                 Set verbosity level of logging: trace, debug, info, warn, err(or), cri

In [15]:
!time ~/marian/build/marian-decoder -m ./model.transformer.phmy/model.npz -v ../g2p-par/vocab/vocab.ph.yml ../g2p-par/vocab/vocab.my.yml --devices 0 < ../g2p-par/test.ph > ../transformer.phmy.hyp.txt

[2026-07-03 20:42:39] [marian] Marian v1.12.0 65bf82ff 2023-02-21 09:56:29 -0800
[2026-07-03 20:42:39] [marian] Running on PS42 as process 35253 with command line:
[2026-07-03 20:42:39] [marian] /home/phantom/marian/build/marian-decoder -m ./model.transformer.phmy/model.npz -v ../g2p-par/vocab/vocab.ph.yml ../g2p-par/vocab/vocab.my.yml --devices 0
[2026-07-03 20:42:39] [config] alignment: ""
[2026-07-03 20:42:39] [config] allow-special: false
[2026-07-03 20:42:39] [config] allow-unk: false
[2026-07-03 20:42:39] [config] authors: false
[2026-07-03 20:42:39] [config] beam-size: 12
[2026-07-03 20:42:39] [config] bert-class-symbol: "[CLS]"
[2026-07-03 20:42:39] [config] bert-mask-symbol: "[MASK]"
[2026-07-03 20:42:39] [config] bert-masking-fraction: 0.15
[2026-07-03 20:42:39] [config] bert-sep-symbol: "[SEP]"
[2026-07-03 20:42:39] [config] bert-train-type-embeddings: true
[2026-07-03 20:42:39] [config] bert-type-vocab-size: 2
[2026-07-03 20:42:39] [config] best-deep: false
[2026-07-03 20:4

**Marian က training/testing အတွက် အရမ်းမြန်ပါတယ်။ အဲဒါကြောင့် ဆရာ ကြိုက်တယ်။**

## Evaluation

In [16]:
!perl /home/phantom/mosesdecoder/scripts/generic/multi-bleu.perl ../g2p-par/test.my < ../transformer.phmy.hyp.txt

BLEU = 76.45, 87.6/78.3/72.7/68.8 (BP=0.999, ratio=0.999, hyp_len=8038, ref_len=8047)
It is not advisable to publish scores from multi-bleu.perl.  The scores depend on your tokenizer, which is unlikely to be reproducible from your paper or consistent across research groups.  Instead you should detokenize then use mteval-v14.pl, which has a standard tokenization.  Scores from multi-bleu.perl can still be used for internal purposes when you have a consistent tokenizer.


Sequence to Sequence နဲ့ Transformer မော်ဒယ်နှစ်ရဲ့ ရလဒ်ကို နှိုင်းယှဉ်ကြည့်တဲ့အခါမှာ Sequence to Sequence မော်ဒယ်ရဲ့ ရလဒ်က ပိုကောင်းတာကို တွေ့ရပါလိမ့်မယ်။  